# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Available Record Sets (@id, name):")
for rs in metadata.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

# For the primary record set, print associated fields and columns by @id
# Let's find the first record set with fields
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_record_set = None
    for rs in metadata.record_sets:
        if rs['@id'] == main_record_set_id:
            main_record_set = rs
            break
    print(f"\nFields for record set {main_record_set_id}:")
    for field in main_record_set.get('fields', []):
        print(f"  - {field['@id']}: {field['name']}")
        # If the field is associated with a column, print the column @id
        if 'column' in field:
            print(f"    column @id: {field['column']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @ids
record_set_ids = [r['@id'] for r in metadata.record_sets]
dataframes = {}

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns of primary record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in record set {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to detect a numeric field in the columns. We'll look for columns with float/int dtype or typical numeric names
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # If dtype detection fails, look for likely numeric columns
        for col in df.columns:
            if any(x in col.lower() for x in ['amount', 'mw', 'mass', 'count', 'coverage', 'percent', 'abundance', 'value', 'peptide','intensity','score']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    continue
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Pick the first numeric field for demonstration
        threshold = df[numeric_field].mean()  # Use the mean as a demonstration threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records\n")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df.loc[:, f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field} (first 5 rows):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field, e.g., based on a 'description', 'accession', or any non-numeric field
        group_candidates = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields were detected in the primary record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_candidates:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    ax.set_xlabel(numeric_field)
    ax.set_ylabel('Count')
    plt.show()

    # If there's a group_field, plot boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.